# Day 3: Build & Deploy a RAG App with Streamlit

**FAMNIT AI Workshop** | 50-minute hands-on guide

In this session you will:
1. Learn the basics of **Streamlit** (Python web framework)
2. Build a **RAG app** with LangChain + ChromaDB
3. Push your code to **GitHub**
4. Deploy your app live on **Render.com**

By the end, you will have a working web app that anyone on the internet can visit!

---
## Part 1: Your First Streamlit App (15 min)
---

### What is Streamlit?

**Streamlit** is a Python framework for building interactive web applications.  
The magic: **you don't need to know HTML, CSS, or JavaScript**. You just write Python.

| Framework | Approach | Best for |
|-----------|----------|----------|
| **Flask / Django** | Full control, but you write HTML templates, routes, forms... | Complex web apps with custom UIs |
| **Streamlit** | Write Python, get a web app instantly | Data apps, dashboards, ML demos |

Streamlit re-runs your entire script from top to bottom every time the user interacts with the app.  
This makes it simple to reason about, but also means you need caching for expensive operations (we will see this later).

### Install Streamlit

Run this in your terminal (not inside a notebook):

```
pip install streamlit
```

To verify the installation:

```
streamlit hello
```

This opens a demo app in your browser. Close it when you are done exploring.

### Hello World

Create a file called **`app.py`** and paste the following code:

```python
import streamlit as st

st.title("Hello, World!")
st.write("My first Streamlit app")

name = st.text_input("What's your name?")
if name:
    st.balloons()
    st.write(f"Hello, {name}!")
```

Then run it from your terminal:

```
streamlit run app.py
```

Your browser will open at **http://localhost:8501**. Try typing your name in the text field!

> **How it works:** Every time you type something, Streamlit re-runs the entire script.  
> The `if name:` block only executes when the text field is not empty.

### Key Streamlit Components

Here are the most useful building blocks you can use in your apps:

| Component | What it does | Example |
|-----------|-------------|---------|
| `st.title()` | Big heading | `st.title("My App")` |
| `st.write()` | Display text, data, tables, charts | `st.write("Hello")` |
| `st.text_input()` | Single-line text field | `name = st.text_input("Name")` |
| `st.text_area()` | Multi-line text field | `text = st.text_area("Paste text")` |
| `st.button()` | Clickable button | `if st.button("Go"): ...` |
| `st.slider()` | Numeric slider | `n = st.slider("Count", 1, 10)` |
| `st.selectbox()` | Dropdown menu | `choice = st.selectbox("Pick", ["A","B"])` |
| `st.sidebar` | Side navigation panel | `st.sidebar.title("Menu")` |
| `st.columns()` | Side-by-side layout | `col1, col2 = st.columns(2)` |
| `st.spinner()` | Loading indicator | `with st.spinner("..."): ...` |
| `st.metric()` | Big number display | `st.metric("Score", 95)` |
| `st.expander()` | Collapsible section | `with st.expander("More"): ...` |

You can explore all components in the [Streamlit docs](https://docs.streamlit.io/develop/api-reference).

---
## Part 2: Adding RAG with LangChain + ChromaDB (20 min)
---

### Install Dependencies

Run this in your terminal:

```
pip install langchain langchain-community langchain-huggingface langchain-text-splitters chromadb sentence-transformers
```

This installs:
- **LangChain** -- framework for chaining AI components together
- **ChromaDB** -- a lightweight vector database that runs locally
- **sentence-transformers** -- pre-trained models that convert text to numerical vectors (embeddings)

### The RAG Pipeline

RAG stands for **Retrieval-Augmented Generation**. Here is the flow of data in our app:

```
Your Documents
      |
      v
  Chunking          (split long texts into small pieces)
      |
      v
  Embeddings        (convert each chunk into a vector of numbers)
      |
      v
  ChromaDB          (store vectors in a searchable database)
      |
      v
  User Query  --->  Semantic Search  --->  Results
  (also embedded)   (find closest vectors)  (most relevant chunks)
```

The key idea: instead of searching for exact keyword matches, we search for **meaning**.  
"What is ML?" will find text about "machine learning" even if the exact phrase "ML" does not appear.

### Step-by-step Code Explanation

The starter app below has six main sections. Let us walk through each one:

**1. Imports and page configuration**
```python
import streamlit as st
import numpy as np

st.set_page_config(page_title="My RAG Knowledge Base", page_icon="...", layout="wide")
```
`set_page_config` must be the first Streamlit command. It sets the browser tab title and page layout.

**2. Define your documents**
```python
DOCUMENTS = [
    """Python is a high-level programming language...""",
    """Machine learning is a subset of AI...""",
    # ... more documents
]
```
This is a simple Python list of strings. Each string is one "document". You will replace these with your own texts.

**3. Load embedding model with `@st.cache_resource`**
```python
@st.cache_resource(show_spinner="Loading embedding model...")
def load_embedding_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
```
The `@st.cache_resource` decorator means: run this function **once**, then reuse the result on every re-run. Without caching, the model would reload every time you type a character!

**4. Build vector store: chunk, embed, store**
```python
@st.cache_resource(show_spinner="Building vector database...")
def build_vector_store(_documents):
    # Split documents into chunks of ~300 characters
    # Embed each chunk using the model
    # Store in ChromaDB
    return vector_store, chunks
```
Also cached. The underscore prefix `_documents` tells Streamlit not to hash this argument (lists are mutable and hard to hash).

**5. Create the search interface**
```python
query = st.text_input("Your question")
results = vector_store.similarity_search_with_score(query, k=3)
```
The user types a query, it gets embedded, and ChromaDB finds the 3 closest chunks.

**6. Display results**
```python
for doc, score in results:
    st.markdown(f"> {doc.page_content}")
```
Each result is shown as a blockquote with a relevance score.

In [ ]:
# ==============================================================
# COMPLETE STARTER APP CODE
# ==============================================================
# Copy everything below into a file called  app.py
# Then run:  streamlit run app.py
# ==============================================================

APP_CODE = '''
"""
RAG Knowledge Base - Starter Template
FAMNIT AI Course - Day 3

A simple Retrieval-Augmented Generation (RAG) app built with
Streamlit, LangChain, and ChromaDB. No API keys needed!

Instructions:
  1. Replace the DOCUMENTS list below with your own texts
  2. Update the app title and description
  3. Run locally:  streamlit run app.py
  4. Deploy to Render (see assignment instructions)
"""

import streamlit as st
import numpy as np

st.set_page_config(
    page_title="My RAG Knowledge Base",
    page_icon="\U0001f50d",
    layout="wide",
)

# ----------------------------------------------------------------
# YOUR DOCUMENTS - Replace these with your own topic!
# Each string is one "document" that will be chunked, embedded,
# and stored in the vector database for semantic search.
# ----------------------------------------------------------------
DOCUMENTS = [
    """Python is a high-level, general-purpose programming language. Its design
philosophy emphasizes code readability with the use of significant
indentation. Python is dynamically typed and garbage-collected. It supports
multiple programming paradigms, including structured, object-oriented and
functional programming.""",

    """Machine learning is a subset of artificial intelligence that focuses on
building systems that learn from data. Instead of being explicitly
programmed, these systems identify patterns in data and make decisions with
minimal human intervention. Common types include supervised learning,
unsupervised learning, and reinforcement learning.""",

    """Streamlit is an open-source Python framework for building interactive web
applications for data science and machine learning. It allows developers to
create apps with just a few lines of Python code, without needing to know
HTML, CSS, or JavaScript. Streamlit apps can be deployed easily to the cloud.""",

    """ChromaDB is an open-source vector database designed for AI applications.
It stores embeddings (numerical representations of data) and allows
efficient similarity search. ChromaDB is lightweight, runs locally, and
integrates well with LangChain and other AI frameworks.""",

    """Retrieval-Augmented Generation (RAG) is a technique that enhances AI
responses by first retrieving relevant information from a knowledge base,
then using that information to generate accurate answers. RAG combines the
power of search (retrieval) with language generation.""",
]

# ----------------------------------------------------------------
# Cached heavy resources (loaded once, reused across reruns)
# ----------------------------------------------------------------

@st.cache_resource(show_spinner="Loading embedding model...")
def load_embedding_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


@st.cache_resource(show_spinner="Building vector database...")
def build_vector_store(_documents: tuple):
    """Chunk documents, embed them, and store in ChromaDB."""
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import Chroma

    # --- Chunking ---
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50,
        separators=["\\n\\n", "\\n", ". ", " ", ""],
    )
    chunks = []
    for doc in _documents:
        chunks.extend(splitter.split_text(doc))

    embeddings = load_embedding_model()

    # --- Store in ChromaDB ---
    vector_store = Chroma.from_texts(
        texts=chunks,
        embedding=embeddings,
        collection_name="knowledge_base",
    )
    return vector_store, chunks


# ----------------------------------------------------------------
# SIDEBAR
# ----------------------------------------------------------------
st.sidebar.title("My RAG App")
page = st.sidebar.radio("Navigate", ["Home", "Search", "Explore Chunks"])

# ----------------------------------------------------------------
# HOME PAGE
# ----------------------------------------------------------------
if page == "Home":
    st.title("My RAG Knowledge Base")
    st.markdown("""
    Welcome! This app lets you **search documents by meaning**, not just keywords.

    ### How it works
    1. **Documents** are split into small chunks
    2. Each chunk is converted to an **embedding** (a vector of numbers)
    3. Chunks are stored in a **vector database** (ChromaDB)
    4. When you search, your query is embedded and compared to all chunks
    5. The most **semantically similar** chunks are returned

    ### Get started
    - Go to **Search** to ask questions
    - Go to **Explore Chunks** to see how documents are split

    ---
    *Built with Streamlit, LangChain, and ChromaDB*
    """)

    st.info(f"Knowledge base contains **{len(DOCUMENTS)} documents**.")


# ----------------------------------------------------------------
# SEARCH PAGE
# ----------------------------------------------------------------
elif page == "Search":
    st.title("Semantic Search")
    st.markdown("Ask a question and the app will find the most relevant chunks from the knowledge base.")

    vector_store, chunks = build_vector_store(tuple(DOCUMENTS))

    query = st.text_input(
        "Your question",
        placeholder="e.g. What is RAG?",
    )
    num_results = st.slider("Number of results", 1, 10, 3)

    if query:
        with st.spinner("Searching..."):
            results = vector_store.similarity_search_with_score(query, k=num_results)

        st.subheader(f"Top {len(results)} results")
        for i, (doc, score) in enumerate(results, 1):
            similarity = max(0, 1 - score)  # rough conversion
            with st.container():
                st.markdown(f"**Result {i}** -- relevance: `{similarity:.2f}`")
                st.markdown(f"> {doc.page_content}")
                st.divider()

    st.markdown("---")
    st.caption("Powered by all-MiniLM-L6-v2 embeddings + ChromaDB")


# ----------------------------------------------------------------
# EXPLORE CHUNKS PAGE
# ----------------------------------------------------------------
elif page == "Explore Chunks":
    st.title("Explore Chunks")
    st.markdown("See how your documents are split into chunks by the recursive text splitter.")

    vector_store, chunks = build_vector_store(tuple(DOCUMENTS))

    st.metric("Total chunks", len(chunks))

    lengths = [len(c) for c in chunks]
    col1, col2, col3 = st.columns(3)
    col1.metric("Avg chunk size", f"{np.mean(lengths):.0f} chars")
    col2.metric("Min chunk size", f"{min(lengths)} chars")
    col3.metric("Max chunk size", f"{max(lengths)} chars")

    st.subheader("All chunks")
    for i, chunk in enumerate(chunks, 1):
        with st.expander(f"Chunk {i} ({len(chunk)} chars)"):
            st.text(chunk)
'''

print("App code is stored in the APP_CODE variable above.")
print("Copy it into a file called app.py and run: streamlit run app.py")

### Try It Locally

Follow these steps:

1. **Copy** the code from the cell above into a file called **`app.py`**  
   (or use the starter file from the `streamlit-starter/` folder)
2. **Open a terminal** in the folder where `app.py` is saved
3. **Run:**
   ```
   streamlit run app.py
   ```
4. Your browser will open at **http://localhost:8501**
5. Navigate to the **Search** page and try queries like:
   - "What is machine learning?"
   - "How does RAG work?"
   - "Tell me about vector databases"

> **Note:** The first run will download the embedding model (~90 MB). This only happens once.

### Exercise: Customize Your App

Now make it your own! **Replace the `DOCUMENTS` list** with texts about something **you** are interested in.

Ideas:
- Your favorite sport (rules, history, famous players)
- A course topic (summaries of key concepts)
- Cooking recipes
- Movie summaries
- Travel destinations
- Song lyrics
- Historical events

Tips:
- Use at least **5 documents** for interesting search results
- Each document should be a paragraph (3-6 sentences)
- The more distinct the topics, the better the semantic search works
- Update the app title in `st.set_page_config()` and `st.title()` to match your topic

---
## Part 3: Push to GitHub (5 min)
---

### Create a GitHub Repository

1. Go to **[github.com](https://github.com)** and log in
2. Click the **"+"** button (top right) and select **"New repository"**
3. Repository name: **`my-rag-app`** (or choose your own name)
4. Set visibility to **Public**
5. **Do NOT** check "Add a README file" (we will push our own code)
6. Click **"Create repository"**

GitHub will show you setup instructions. We will use the commands below instead.

### Push Your Code

Make sure your project folder contains these **4 files**:

| File | Purpose |
|------|---------|
| `app.py` | Your Streamlit application |
| `requirements.txt` | Python dependencies for Render to install |
| `render.yaml` | Deployment configuration for Render |
| `.gitignore` | Files to exclude from version control |

If you do not have `requirements.txt` yet, create it with this content:
```
streamlit>=1.30.0
langchain>=0.2.0
langchain-community>=0.2.0
langchain-huggingface>=0.0.3
langchain-text-splitters>=0.2.0
chromadb>=0.4.22
sentence-transformers>=2.2.2
numpy>=1.24.0
pandas>=2.0.0
```

If you do not have `.gitignore` yet, create it with:
```
__pycache__/
*.pyc
.env
chroma_data/
.streamlit/
```

Now run these commands in your terminal:

```bash
# Navigate to your project folder
cd my-rag-app

# Initialize a git repository
git init

# Stage your files
git add app.py requirements.txt render.yaml .gitignore

# Create your first commit
git commit -m "Initial RAG app"

# Connect to your GitHub repository (replace YOUR_USERNAME with your GitHub username)
git remote add origin https://github.com/YOUR_USERNAME/my-rag-app.git

# Rename the branch to main and push
git branch -M main
git push -u origin main
```

> **If prompted for credentials:** GitHub now requires a Personal Access Token instead of your password.  
> Go to GitHub Settings > Developer settings > Personal access tokens > Generate new token.

---
## Part 4: Deploy to Render.com (10 min)
---

### What is Render?

**[Render](https://render.com)** is a cloud platform for deploying web applications.

Why Render?
- **Free tier** available (no credit card needed)
- **Connects directly** to your GitHub repository
- **Auto-deploys** every time you push new code
- Simple setup -- no Docker or complex configuration required

Other options exist (Heroku, Railway, Streamlit Cloud), but Render has a generous free tier and straightforward setup.

### Step-by-step Render Deployment

Follow these steps carefully:

**1. Create a Render account**
- Go to **[render.com](https://render.com)**
- Click **"Get Started for Free"**
- Choose **"Sign in with GitHub"** (this makes connecting your repo easier)

**2. Create a new Web Service**
- From your Render dashboard, click **"New"** (top right)
- Select **"Web Service"**

**3. Connect your repository**
- Search for your **`my-rag-app`** repository
- Click **"Connect"**

**4. Configure the service**
- Render will detect your **`render.yaml`** file automatically
- It will fill in the build command and start command for you
- Make sure the **Free** plan is selected

**5. Deploy**
- Click **"Create Web Service"**
- Wait **5-10 minutes** for the first build (it needs to install all packages and download the embedding model)
- You can watch the build progress in the Render logs

**6. Your app is live!**
- Once the build succeeds, your app will be available at:  
  **`https://my-rag-app.onrender.com`**  
  (the exact URL depends on the name you chose)
- Share this URL with your classmates!

### Important Notes About the Free Tier

- **Sleeping:** Free tier apps "sleep" after **15 minutes** of inactivity. The first visit after sleeping takes **~30 seconds** to wake up. This is normal.
- **First build:** The initial build downloads the embedding model (~90 MB) and installs all packages. Be patient -- this can take 5-10 minutes.
- **If the build fails:** Check the Render logs. The most common cause is a missing package in `requirements.txt`.
- **Updating your app:** Just `git add`, `git commit`, and `git push`. Render will **automatically** detect the new code and redeploy.
- **Monthly limits:** Free tier gives you 750 hours/month, which is plenty for a personal project.

### The render.yaml File

This file tells Render how to build and run your app. Here is what each line means:

```yaml
services:
  - type: web                    # This is a web service (not a background worker)
    name: my-rag-app             # The name of your service (appears in the URL)
    runtime: python              # Use the Python runtime
    buildCommand: pip install -r requirements.txt   # Run during build to install packages
    startCommand: streamlit run app.py --server.port=$PORT --server.address=0.0.0.0 --server.headless=true
    #              |              |                    |                        |
    #              Run Streamlit  Use Render's port    Listen on all IPs       No browser popup
    envVars:
      - key: PYTHON_VERSION
        value: "3.11"            # Use Python 3.11
      - key: STREAMLIT_SERVER_ENABLE_CORS
        value: "false"           # Disable CORS (not needed for single-page apps)
      - key: STREAMLIT_SERVER_ENABLE_XSRF_PROTECTION
        value: "false"           # Disable XSRF protection (simplifies deployment)
```

The most important part is the `startCommand`. The `--server.port=$PORT` flag tells Streamlit to use whatever port Render assigns (instead of the default 8501).

### Troubleshooting

If something goes wrong, check this table first:

| Problem | Solution |
|---------|----------|
| Build fails | Check `requirements.txt` has all packages. Look at the Render build logs for the exact error. |
| App crashes on start | Check the Render **logs** tab for error messages. Common cause: a typo in `app.py`. |
| "ModuleNotFoundError" | Add the missing module to `requirements.txt`, commit, and push. |
| App is slow on first load | Normal! Free tier needs to wake up (~30s) and may download the embedding model. |
| Port error | Make sure `startCommand` in `render.yaml` includes `--server.port=$PORT`. |
| "Address already in use" | Make sure `--server.address=0.0.0.0` is in the start command. |
| Changes not showing up | Did you `git push`? Check that the Render deploy was triggered (dashboard > Deploys). |
| GitHub repo not found | Make sure the repo is **Public**, or grant Render access to your private repos. |

---
## Congratulations!
---

You now have a **live RAG application** on the internet!

**What you accomplished today:**
- Built an interactive web app with **Streamlit**
- Implemented **semantic search** using embeddings and ChromaDB
- Pushed your code to **GitHub**
- Deployed to the cloud with **Render**

**Share your app URL** with your classmates and try searching each other's knowledge bases!

**Next steps to explore on your own:**
- Add more documents to your knowledge base
- Experiment with different `chunk_size` and `chunk_overlap` values
- Try adding a file uploader (`st.file_uploader`) so users can add their own documents
- Connect a language model (like Ollama or an API) to generate answers from the retrieved chunks

---
*FAMNIT AI Workshop - Day 3*